<a href="https://colab.research.google.com/github/estebanolroberto/TFM_SERGIO-ROBERTO-UNIR/blob/main/GetMemecoinsData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import requests
import pandas as pd

# Lista de monedas y sus IDs en CoinGecko
coins = {
    'pepe': 'pepe',
    'doge': 'dogecoin',
    'shiba': 'shiba-inu'
}

# Iterar por cada moneda
for name, coingecko_id in coins.items():
    print(f"Descargando datos de {name.upper()}...")

    url = f'https://api.coingecko.com/api/v3/coins/{coingecko_id}/market_chart'
    params = {
        'vs_currency': 'usd',
        'days': '365'
    }

    response = requests.get(url, params=params)
    data = response.json()

    # Crear DataFrame
    prices = data['prices']
    df = pd.DataFrame(prices, columns=['timestamp', 'price'])

    # Convertir timestamp a fecha (sin hora)
    df['date'] = pd.to_datetime(df['timestamp'], unit='ms').dt.date

    # Agrupar por fecha y quedarnos con el promedio
    daily_df = df.groupby('date', as_index=False).mean()

    # Guardar como CSV con separador ";"
    filename = f'{name}_prices_last_365_days_mean.csv'
    daily_df.to_csv(filename, index=False, sep=';')
    print(f"{filename} guardado correctamente.\n")


Descargando datos de PEPE...
pepe_prices_last_365_days_mean.csv guardado correctamente.

Descargando datos de DOGE...
doge_prices_last_365_days_mean.csv guardado correctamente.

Descargando datos de SHIBA...
shiba_prices_last_365_days_mean.csv guardado correctamente.



In [12]:
import requests
import pandas as pd

# Diccionario con los IDs de CoinGecko
coins = {
    'pepe': 'pepe',
    'doge': 'dogecoin',
    'shiba': 'shiba-inu'
}

# Loop para cada moneda
for name, coingecko_id in coins.items():
    print(f"Procesando {name.upper()}...")

    url = f'https://api.coingecko.com/api/v3/coins/{coingecko_id}/ohlc'
    params = {
        'vs_currency': 'usd',
        'days': '365'  # CoinGecko permite 1, 7, 14, 30, 90, 365
    }

    response = requests.get(url, params=params)
    data = response.json()

    if not data:
        print(f"❌ No se pudieron obtener datos para {name.upper()}.")
        continue

    # Crear DataFrame
    df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close'])

    # Convertir timestamp a fecha
    df['date'] = pd.to_datetime(df['timestamp'], unit='ms').dt.date

    # Calcular métricas de volatilidad
    df['range'] = df['high'] - df['low']
    df['pct_change'] = (df['close'] - df['open']) / df['open'] * 100
    df['rel_volatility'] = (df['high'] - df['low']) / ((df['high'] + df['low']) / 2)

    # Redondear para evitar problemas de formato
    df['range'] = df['range'].round(10)
    df['pct_change'] = df['pct_change'].round(6)
    df['rel_volatility'] = df['rel_volatility'].round(6)

    # Seleccionar columnas relevantes
    final_df = df[['date', 'open', 'high', 'low', 'close', 'range', 'pct_change', 'rel_volatility']]

    # Guardar como CSV con separador ";" y decimal ","
    filename = f'{name}_volatility_last_365_days.csv'
    final_df.to_csv(filename, index=False, sep=';', decimal=',')

    print(f"✅ Archivo guardado: {filename}\n")


Procesando PEPE...
✅ Archivo guardado: pepe_volatility_last_365_days.csv

Procesando DOGE...
✅ Archivo guardado: doge_volatility_last_365_days.csv

Procesando SHIBA...
✅ Archivo guardado: shiba_volatility_last_365_days.csv

